# Offline network and fixed Top100-subset comparison

Run from the package root (`Path.cwd()`) after `01_export_final_edges.ipynb`. Recomputation reads only packaged native `regulons.json` and exported `final_edges.tsv.gz`, using pandas and the Python standard library. It does not import SpaGRN/pySCENIC, read motifs, access H5AD/AUCell matrices, or train a network. `workflow_manifest.json` supplies run labels, seeds, and the package-relative JSON/edge paths; its external large-file paths are not opened.

Network identities are `(regulon, target)` pairs; native JSON defines memberships. Self edges and signed weights remain present. Rankings sort **weight descending, target ascending**, without absolute weights. The Top100 subset is selected once for each repeat: a regulon must have at least 100 targets in **both** full and repeat. The same regulons are then evaluated at k = 20, 40, 60, 80, 100. Each Jaccard is `|Top-k(full) ∩ Top-k(repeat)| / |Top-k(full) ∪ Top-k(repeat)|`.

The historical tables in `data/figure_inputs` are used only for validation after recomputation. They were copied from `bin100/robustness/results`; they are not inputs to the calculations below. The output tables in `data/analysis_recomputed` and executed cell outputs are produced during this packaging run. This validates reproduced numerical definitions, not biological correctness or the original inference.

In [1]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import sys
import pandas as pd
from IPython.display import display

PACKAGE = Path.cwd()
assert (PACKAGE / "manifests/workflow_manifest.json").is_file(), "Run from the reproducibility package root."
analysis_started_utc = datetime.now(timezone.utc).isoformat()
manifest = json.loads((PACKAGE / "manifests/workflow_manifest.json").read_text())
runs = manifest["runs"]
TOP_K = (5, 10, 20, 30, 40, 50, 60, 80, 100)
TOP100_K = (20, 40, 60, 80, 100)
native = {}
edges = {}
input_checks = []
for info in [manifest["full"]] + runs:
    run = info["run"]
    json_path = PACKAGE / info["regulons"]
    edge_path = PACKAGE / info["final_edges"]
    assert not Path(info["regulons"]).is_absolute()
    assert not Path(info["final_edges"]).is_absolute()
    native[run] = json.loads(json_path.read_text())
    edges[run] = pd.read_csv(edge_path, sep="\t", float_precision="round_trip")
    assert list(edges[run].columns) == ["run", "regulon", "TF", "target", "weight"]
    assert set(edges[run]["run"]) == {run}
    assert not edges[run].duplicated(["regulon", "target"]).any()
    assert edges[run]["weight"].notna().all()
    observed = set(zip(edges[run]["regulon"], edges[run]["target"]))
    expected = {(name, target) for name, targets in native[run].items() for target in targets}
    assert observed == expected
    input_checks.append({"run": run, "n_regulons": len(native[run]), "n_edges": len(observed),
                         "json_sha256": hashlib.sha256(json_path.read_bytes()).hexdigest(),
                         "final_edges_sha256": hashlib.sha256(edge_path.read_bytes()).hexdigest(),
                         "membership_exact": True})
full_regulons = native["full"]
full_edges = edges["full"]
full_regulon_set = set(full_regulons)
full_edge_set = set(zip(full_edges["regulon"], full_edges["target"]))
full_by_regulon = {name: set(targets) for name, targets in full_regulons.items()}
full_ranked = {
    name: group.sort_values(["weight", "target"], ascending=[False, True])["target"].tolist()
    for name, group in full_edges.groupby("regulon")
}
assert len(full_regulon_set) == 260 and len(full_edge_set) == 68492
display(pd.DataFrame(input_checks).drop(columns=["json_sha256", "final_edges_sha256"]))

,run,n_regulons,n_edges,membership_exact
0,full,260,68492,True
1,rep01,220,33270,True
2,rep02,221,33212,True
3,rep03,213,33745,True
4,rep04,218,33381,True
5,rep05,212,34063,True


In [2]:
run_rows = []
regulon_rows = []
topk_rows = []
for info in runs:
    run, seed = info["run"], info["seed"]
    sample = native[run]
    sample_edges = edges[run]
    sample_regulon_set = set(sample)
    sample_edge_set = set(zip(sample_edges["regulon"], sample_edges["target"]))
    shared_edges = len(full_edge_set & sample_edge_set)
    shared_regulons = len(full_regulon_set & sample_regulon_set)
    run_rows.append({
        "run": run, "seed": seed, "n_regulons": len(sample_regulon_set),
        "n_edges": len(sample_edge_set), "shared_regulons": shared_regulons,
        "regulon_retention": shared_regulons / len(full_regulon_set),
        "regulon_jaccard": shared_regulons / len(full_regulon_set | sample_regulon_set),
        "shared_edges": shared_edges, "edge_retention": shared_edges / len(full_edge_set),
        "edge_jaccard": shared_edges / len(full_edge_set | sample_edge_set),
        "lost_regulons": len(full_regulon_set - sample_regulon_set),
        "new_regulons": len(sample_regulon_set - full_regulon_set),
    })
    sample_by_regulon = {name: set(targets) for name, targets in sample.items()}
    sample_ranked = {
        name: group.sort_values(["weight", "target"], ascending=[False, True])["target"].tolist()
        for name, group in sample_edges.groupby("regulon")
    }
    all_names = sorted(full_regulon_set | sample_regulon_set)
    current_rows = []
    for name in all_names:
        full_targets = full_by_regulon.get(name, set())
        sample_targets = sample_by_regulon.get(name, set())
        shared = len(full_targets & sample_targets)
        full_rank = full_ranked.get(name, [])
        sample_rank = sample_ranked.get(name, [])
        top_jaccards = {}
        for k in TOP_K:
            if len(full_rank) >= k and len(sample_rank) >= k:
                full_top, sample_top = set(full_rank[:k]), set(sample_rank[:k])
                top_jaccards[k] = len(full_top & sample_top) / len(full_top | sample_top)
            else:
                top_jaccards[k] = float("nan")
        current_rows.append({
            "run": run, "seed": seed, "regulon": name,
            "status": "retained" if name in full_regulon_set and name in sample_regulon_set else ("missing" if name in full_regulon_set else "new"),
            "n_targets_full": len(full_targets), "n_targets_sample": len(sample_targets),
            "shared_targets": shared,
            "target_retention": shared / len(full_targets) if full_targets else float("nan"),
            "target_jaccard": shared / len(full_targets | sample_targets) if full_targets | sample_targets else float("nan"),
            **{f"top{k}_jaccard": top_jaccards[k] for k in TOP_K},
        })
    regulon_rows.extend(current_rows)
    for k in TOP_K:
        eligible = [row[f"top{k}_jaccard"] for row in current_rows
                    if row["n_targets_full"] >= k and row["n_targets_sample"] >= k]
        reference_eligible = sum(len(targets) >= k for targets in full_by_regulon.values())
        topk_rows.append({"run": run, "k": k, "eligible_regulons": len(eligible),
                          "reference_eligible_regulons": reference_eligible,
                          "coverage": len(eligible) / reference_eligible,
                          "median_jaccard": pd.Series(eligible, dtype="float64").median()})
network_runs = pd.DataFrame(run_rows)
network_regulons = pd.DataFrame(regulon_rows)
network_topk = pd.DataFrame(topk_rows)
numeric_columns = [column for column in network_runs.select_dtypes(include="number").columns if column != "seed"]
network_summary = network_runs[numeric_columns].agg(["median", lambda values: values.quantile(0.25), lambda values: values.quantile(0.75), "min", "max"]).T
network_summary.columns = ["median", "q25", "q75", "min", "max"]
network_summary = network_summary.reset_index().rename(columns={"index": "metric"})
assert tuple(network_runs["n_regulons"]) == (220, 221, 213, 218, 212)
assert tuple(network_runs["shared_regulons"]) == (183, 181, 181, 178, 175)
display(network_runs)
display(network_summary)

,run,seed,n_regulons,n_edges,shared_regulons,regulon_retention,regulon_jaccard,shared_edges,edge_retention,edge_jaccard,lost_regulons,new_regulons
0,rep01,20261016,220,33270,183,0.703846,0.616162,21336,0.311511,0.265287,77,37
1,rep02,20261017,221,33212,181,0.696154,0.603333,21393,0.312343,0.266377,79,40
2,rep03,20261018,213,33745,181,0.696154,0.619863,21234,0.310022,0.262138,79,32
3,rep04,20261019,218,33381,178,0.684615,0.593333,20896,0.305087,0.258049,82,40
4,rep05,20261020,212,34063,175,0.673077,0.589226,21270,0.310547,0.261672,85,37


,metric,median,q25,q75,min,max
0,n_regulons,218.000000,213.000000,220.000000,212.000000,221.000000
1,n_edges,33381.000000,33270.000000,33745.000000,33212.000000,34063.000000
2,shared_regulons,181.000000,178.000000,181.000000,175.000000,183.000000
3,regulon_retention,0.696154,0.684615,0.696154,0.673077,0.703846
4,regulon_jaccard,0.603333,0.593333,0.616162,0.589226,0.619863
5,shared_edges,21270.000000,21234.000000,21336.000000,20896.000000,21393.000000
6,edge_retention,0.310547,0.310022,0.311511,0.305087,0.312343
7,edge_jaccard,0.262138,0.261672,0.265287,0.258049,0.266377
8,lost_regulons,79.000000,79.000000,82.000000,77.000000,85.000000
9,new_regulons,37.000000,37.000000,40.000000,32.000000,40.000000


In [3]:
subset = network_regulons.loc[(network_regulons["n_targets_full"] >= 100) &
                               (network_regulons["n_targets_sample"] >= 100)]
network_topk_top100_subset_per_regulon = subset.melt(
    id_vars=["run", "seed", "regulon", "n_targets_full", "n_targets_sample"],
    value_vars=[f"top{k}_jaccard" for k in TOP100_K], var_name="k", value_name="jaccard")
network_topk_top100_subset_per_regulon["k"] = network_topk_top100_subset_per_regulon["k"].str.extract(r"top(\d+)_jaccard", expand=False).astype(int)
network_topk_top100_subset_per_regulon = network_topk_top100_subset_per_regulon.sort_values(["run", "k", "regulon"]).reset_index(drop=True)
network_topk_top100_subset = network_topk_top100_subset_per_regulon.groupby(["run", "seed", "k"], sort=True).agg(
    eligible_regulons=("regulon", "size"), n_valid=("jaccard", "count"),
    mean_jaccard=("jaccard", "mean"), sd_jaccard=("jaccard", "std"),
    median_jaccard=("jaccard", "median"),
    q25=("jaccard", lambda values: values.quantile(0.25)),
    q75=("jaccard", lambda values: values.quantile(0.75))).reset_index()
network_topk_top100_subset["reference_eligible_regulons"] = sum(len(targets) >= 100 for targets in full_by_regulon.values())
network_topk_top100_subset["coverage"] = network_topk_top100_subset["eligible_regulons"] / network_topk_top100_subset["reference_eligible_regulons"]
assert tuple(subset.groupby("run").size()) == (67, 60, 61, 64, 62)
assert network_topk_top100_subset["reference_eligible_regulons"].eq(116).all()
assert network_topk_top100_subset["eligible_regulons"].eq(network_topk_top100_subset["n_valid"]).all()
for run, group in network_topk_top100_subset_per_regulon.groupby("run"):
    name_sets = [set(rows["regulon"]) for _, rows in group.groupby("k")]
    assert all(names == name_sets[0] for names in name_sets)
output_tables = {
    "network_runs.tsv": network_runs,
    "network_regulons.tsv": network_regulons,
    "network_topk.tsv": network_topk,
    "network_summary.tsv": network_summary,
    "network_topk_top100_subset_per_regulon.tsv": network_topk_top100_subset_per_regulon,
    "network_topk_top100_subset.tsv": network_topk_top100_subset,
}
output_dir = PACKAGE / "data/analysis_recomputed"
output_dir.mkdir(exist_ok=True)
for filename, table in output_tables.items():
    table.to_csv(output_dir / filename, sep="\t", index=False)
display(network_topk_top100_subset.pivot(index="k", columns="run", values="median_jaccard"))
display(network_topk_top100_subset.pivot(index="k", columns="run", values="eligible_regulons"))

run,rep01,rep02,rep03,rep04,rep05
k,,,,,
20,0.538462,0.600000,0.600000,0.509972,0.538462
40,0.538462,0.553544,0.568627,0.538462,0.538462
60,0.518987,0.558442,0.558442,0.518987,0.538462
80,0.495327,0.545930,0.553398,0.488404,0.516622
100,0.459854,0.520935,0.538462,0.476035,0.515152


run,rep01,rep02,rep03,rep04,rep05
k,,,,,
20,67,60,61,64,62
40,67,60,61,64,62
60,67,60,61,64,62
80,67,60,61,64,62
100,67,60,61,64,62


In [4]:
comparisons = []
for filename, keys in [
    ("network_runs.tsv", ["run"]),
    ("network_regulons.tsv", ["run", "regulon"]),
    ("network_topk_top100_subset.tsv", ["run", "k"]),
    ("network_topk_top100_subset_per_regulon.tsv", ["run", "k", "regulon"]),
]:
    historical_path = PACKAGE / "data/figure_inputs" / filename
    historical = pd.read_csv(historical_path, sep="\t")
    recomputed = output_tables[filename]
    assert not historical.duplicated(keys).any()
    assert not recomputed.duplicated(keys).any()
    historical = historical.sort_values(keys).set_index(keys)
    recomputed = recomputed.sort_values(keys).set_index(keys)
    assert historical.index.equals(recomputed.index), f"Historical key mismatch: {filename}"
    assert list(historical.columns) == list(recomputed.columns), f"Column mismatch: {filename}"
    pd.testing.assert_frame_equal(historical, recomputed, check_exact=False, rtol=1e-12, atol=1e-12)
    numeric = historical.select_dtypes(include="number").columns
    max_difference = float((historical[numeric] - recomputed[numeric]).abs().max().max())
    comparisons.append({
        "file": filename, "status": "passed", "keys": keys, "rows_compared": len(historical),
        "key_sets_exact": True, "missing_value_positions_exact": bool(historical.isna().equals(recomputed.isna())),
        "historical_source": "bin100/robustness/results/" + filename,
        "historical_packaged_path": str(historical_path.relative_to(PACKAGE)),
        "historical_sha256": hashlib.sha256(historical_path.read_bytes()).hexdigest(),
        "recomputed_packaged_path": "data/analysis_recomputed/" + filename,
        "numeric_rtol": 1e-12, "numeric_atol": 1e-12, "max_absolute_numeric_difference": max_difference,
    })
analysis_finished_utc = datetime.now(timezone.utc).isoformat()
validation_path = PACKAGE / "manifests/grn_validation.json"
validation = json.loads(validation_path.read_text())
assert validation["export"]["status"] == "passed"
validation["offline_analysis"] = {
    "status": "passed", "actual_execution": True,
    "notebook": "notebooks/analysis/02_network_topk.ipynb",
    "started_utc": analysis_started_utc, "finished_utc": analysis_finished_utc,
    "python": sys.executable, "pandas_version": pd.__version__,
    "computation_inputs": "package-native regulons.json and exported final_edges.tsv.gz only; workflow_manifest supplies run/seed/path metadata",
    "input_checks": input_checks,
    "network_edge_key": ["regulon", "target"],
    "ranking": {"columns": ["weight", "target"], "ascending": [False, True], "absolute_weight": False},
    "top100_subset": {"selection": "both full and repeat native target counts >= 100; fixed once per repeat", "k": list(TOP100_K), "regulons_per_repeat": {run: int(count) for run, count in subset.groupby("run").size().items()}, "reference_eligible": 116},
    "comparisons": comparisons,
    "output_files": [{"path": "data/analysis_recomputed/" + filename, "rows": len(table), "sha256": hashlib.sha256((output_dir / filename).read_bytes()).hexdigest()} for filename, table in output_tables.items()],
    "external_large_files_opened": False, "training_performed": False,
    "verification_boundary": "Native membership/export consistency and network/Top-k numerical agreement; no inference rerun or biological validation.",
}
validation_path.write_text(json.dumps(validation, indent=2, allow_nan=False) + "\n")
display(pd.DataFrame(comparisons)[["file", "status", "rows_compared", "key_sets_exact", "max_absolute_numeric_difference"]])
display(pd.DataFrame([{"status": "passed", "actual_execution": True, "started_utc": analysis_started_utc, "finished_utc": analysis_finished_utc, "python": sys.executable, "pandas_version": pd.__version__, "training_performed": False}]))

,file,status,rows_compared,key_sets_exact,max_absolute_numeric_difference
0,network_runs.tsv,passed,5,True,5.551115e-17
1,network_regulons.tsv,passed,1486,True,1.110223e-16
2,network_topk_top100_subset.tsv,passed,25,True,8.326673e-17
3,network_topk_top100_subset_per_regulon.tsv,passed,1570,True,1.110223e-16


,status,actual_execution,started_utc,finished_utc,python,pandas_version,training_performed
0,passed,True,2026-11-07T06:29:59.532613+00:00,2026-11-07T06:30:00.813569+00:00,/opt/miniforge/bin/python,2.2.3,False
